In [0]:
# Gold Layer Integrity and Uniqueness Checks
dbutils.widgets.text("catalog", "main", "Catalog Name")
dbutils.widgets.text("schema", "lab_data", "Schema Name")

catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")

print(f"Validating Gold Layer in: {catalog}.{schema}")

# Check 1: Uniqueness of primary key in dim_movies
df_movies = spark.table(f"{catalog}.{schema}.dim_movies")
total_movies = df_movies.count()
unique_movie_ids = df_movies.select("movie_id").distinct().count()

assert total_movies == unique_movie_ids, f"Gold Error: Duplicate movie_id found! {total_movies} total vs {unique_movie_ids} unique"
print(f"PASS: dim_movies primary keys are strictly unique ({total_movies} records)")

# Check 2: Uniqueness of genre_id in dim_genres
df_genres = spark.table(f"{catalog}.{schema}.dim_genres")
total_genres = df_genres.count()
unique_genre_ids = df_genres.select("genre_id").distinct().count()

assert total_genres == unique_genre_ids, f"Gold Error: Duplicate genre_id found!"
print(f"PASS: dim_genres primary keys are strictly unique ({total_genres} records)")

# Check 3: Check for unwanted NULLs in key analytical dimensions
null_titles = df_movies.filter("title IS NULL OR length(trim(title)) = 0").count()
assert null_titles == 0, f"Gold Error: Found {null_titles} blank titles in dim_movies"
print("PASS: 0 blank titles in dim_movies")